In [15]:
import csv
import importlib
import json
import os
import random
import sys
import torch
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from time import sleep
from collections import deque, defaultdict
from itertools import count
from typing import Any, Dict, Counter, List

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

from importnb import Notebook
with Notebook():
    from Labs.LatencyModel import LatencyModel, MultiDULatencyModel
    from Labs.Policy import DrlPolicy
    from Labs.CacheEngine import CacheEngineEnv
    from Labs.UserRequest import UserRequestEvents
    from Labs.EnvWrapper import EnvWrapper

from RL.Networks import QNetwork, MultiHeadQNetwork
from RL.Buffers import ReplayBuffer, NStepReplayBuffer
from RL.Adapters import FeatureAdapter, NetworkAdapter
from RL.FocusWorkers import BaseWorker, EnhWorker, FocusWorker
from RL.A2CWorker import A2CWorker

import Common.config as config
import Common.datatypes as datatypes
import Common.debugger as debugger
import Common.utils as utils
import Core.builders as builders

importlib.reload(builders)
importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(debugger)
importlib.reload(utils)

<module 'Common.utils' from '/home/eduardo/Workspace/CacheVideoPredict360/Sources/Common/utils.py'>

In [16]:
# The state and action dimensions are derived from the cache layout and the
# observation design used by the agent.
cfg = config.Config()
cfg.filename = \
    f"focus_eps{cfg.epsilon_start}_" \
    f"lrdecay{cfg.learning_rate_decay}_" \
    f"gamma{cfg.gamma}.csv"

# Default ranking depth for reporting and reward shaping.
worst_k = int(getattr(cfg, "worst_k", 20))
top_k = int(getattr(cfg, "topk_content_plot_k", 20))
popularity_reward_scale = getattr(cfg, "popularity_reward_scale", 1.0)

cfg.state_dim = worst_k * 2 + 7
cfg.action_dim = worst_k + 1

debugger = debugger.debug

In [17]:
class NetworkAdapter:
    def __init__(self, cfg: Any, env: Any, feature_adapter: Any):
        self.env = env
        self.cfg = cfg
        self.features = feature_adapter

        self.C = self.cfg.cache_size  # paper's cache capacity (videos)
        self.k = self.cfg.viewport    # paper's tiles per video (enhancement)
        
    def build_observation(self, tile_idx, video, tile=None) -> np.ndarray:

        cache = self.env.mec_cache.policy.cache

        x_s = np.zeros(len(cache), dtype=np.float32)
        x_l = np.zeros(len(cache), dtype=np.float32)

        for idx, (v, t) in enumerate(cache):
            if t == -1:
                x_s[idx] = self.features.video_freq_short.get(v, 0) / self.features.video_hist_short.maxlen
                x_l[idx] = self.features.video_freq_long.get(v, 0) / self.features.video_hist_long.maxlen
            else:
                x_s[idx] = self.features.tile_freq_short.get((v, t), 0) / self.features.tile_hist_short.maxlen
                x_l[idx] = self.features.tile_freq_long.get((v, t), 0) / self.features.tile_hist_long.maxlen

        if tile is None:
            y_s = np.array(
                [self.features.video_freq_short.get(video, 0) / self.features.video_hist_short.maxlen], 
                dtype=np.float32
            )
            y_l = np.array(
                [self.features.video_freq_long.get(video, 0) / self.features.video_hist_long.maxlen], 
                dtype=np.float32
            )
        else:
            y_s = np.array(
                [self.features.tile_freq_short.get((video, tile), 0) / self.features.tile_hist_short.maxlen], 
                dtype=np.float32
            )
            y_l = np.array(
                [self.features.tile_freq_long.get((video, tile), 0) / self.features.tile_hist_long.maxlen], 
                dtype=np.float32
            )
  
        pos_idx = np.argsort(x_l)[:worst_k]

        x_s = x_s[pos_idx]
        x_l = x_l[pos_idx]
        pos_idx = pos_idx + 1
        
        step_one_hot = int(tile is None)
        request_kind = np.zeros(self.cfg.viewport + 1, dtype=np.float32)
        if tile_idx is None:
            request_kind[0] = 1.0
        else:
            request_kind[tile_idx + 1] = 1.0

        return np.concatenate(
            [x_s, x_l, y_s, y_l, request_kind], axis=0
        ), np.concatenate([[0], pos_idx], axis=0)

    def scaled_freq(self, count, hist_maxlen, alpha=10.0):
        if hist_maxlen <= 0:
            return 0.0
        return np.log1p(alpha * count) / np.log1p(alpha * hist_maxlen)

    def reset(self):
        obs, info = self.env.reset()
        return obs, info
    
    def env_is_done(self) -> bool:
        return self.env.users_env.all_users_done()

In [ ]:
def _append_csv_row(csv_path: str, fieldnames: list[str], row: dict) -> None:
    write_header = not os.path.exists(csv_path)
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def save_episode_metrics(
    metrics_dir: str,
    ep: int,
    total_reward: float,
    cache_hits: int,
    cache_misses: int,
    agent,
):
    csv_path = os.path.join(metrics_dir, 'episode_metrics.csv')
    fieldnames = [
        'episode',
        'total_reward',
        'cache_hits',
        'cache_misses',
        'hit_rate',
        'epsilon',
        'lr',
    ]

    row = {
        'episode': ep,
        'total_reward': round(float(total_reward), 2),
        'cache_hits': cache_hits,
        'cache_misses': cache_misses,
        'hit_rate': float(cache_hits) / float(cache_hits + cache_misses + 1e-9),
        'epsilon': round(float(agent.epsilon), 6) if agent else None,
        'lr': float(agent.scheduler.get_last_lr()[0]) if agent else None,
    }
    _append_csv_row(csv_path, fieldnames, row)

def save_step_metrics(
    metrics_dir: str,
    episode: int,
    episode_step: int,
    global_step: int,
    reward: float,
    agent,
    train_metrics: dict | None,
):
    csv_path = os.path.join(metrics_dir, 'step_metrics.csv')
    fieldnames = [
        'episode',
        'episode_step',
        'global_step',
        'reward',
        'epsilon',
        'lr',
        'train_loss',
        'actor_loss',
        'critic_loss'
    ]

    row = {
        'episode': episode,
        'episode_step': episode_step,
        'global_step': global_step,
        'reward': float(reward),
        'epsilon': round(float(agent.epsilon), 6) if agent else None,
        'lr': float(agent.scheduler.get_last_lr()[0]) if agent else None,
        'train_loss': None,
        'actor_loss': None,
        'critic_loss': None
    }

    if train_metrics is not None:
        row.update({
            'train_loss': train_metrics.get('train_loss'),
            'actor_loss': train_metrics.get('actor_loss'),
            'critic_loss': train_metrics.get('critic_loss'),
    })

    _append_csv_row(csv_path, fieldnames, row)

def update_metrics(info: dict, reward: float) -> tuple[float, int, int, int, int]:
    enh_hits = info.get("enh_layer_hits", 0)
    base_hits = info.get("base_layer_hits", 0)
    enh_misses = info.get("enh_layer_misses", 0)
    base_misses = info.get("base_layer_misses", 0)

    return reward, base_hits, base_misses, enh_hits, enh_misses

def log_selection_debug(transition, missing, backhaul_usage):
    if transition[0] is not None:
        debugger.log("base_action", transition[0]["action"])
        debugger.log("base_value", transition[0]["value"])
        debugger.log("base_prob", transition[0]["prob"])
        debugger.log("base_entropy", transition[0]["entropy"])

    for idx, t in enumerate(transition[1:]):
        if t is not None:
            debugger.log(f"enh_{idx}_action", t["action"])
            debugger.log(f"enh_{idx}_value", t["value"])
            debugger.log(f"enh_{idx}_prob", t["prob"])
            debugger.log(f"enh_{idx}_entropy", t["entropy"])

    debugger.log("backhaul_usage", backhaul_usage)
    debugger.log("base_layer_miss", missing[0])
    for i, is_missing in enumerate(missing[1:], start=1):
        debugger.log(f"enh_layer_missing_{i}", is_missing)

In [19]:
def save_episode_models(debug_path: str, episode: int, agent) -> str | None:
    models_dir = os.path.join(debug_path, "models")
    os.makedirs(models_dir, exist_ok=True)

    network_params = {"episode": episode, "networks": {}}

    for net_name in ("q_network", "policy_net", "actor", "critic", "model", "network"):
        net = getattr(agent, net_name, None)
        if net is not None and hasattr(net, "state_dict"):
            network_params["networks"][net_name] = {
                k: v.detach().cpu().tolist() for k, v in net.state_dict().items()
            }

    if not network_params["networks"]:
        return None

    json_path = os.path.join(models_dir, f"network_params_ep{episode}.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(network_params, f, indent=2)

    return json_path

In [20]:
def _content_key(video: int, tile: int | None) -> tuple[int, int]:
    return (int(video), -1 if tile is None else int(tile))

def _content_label(content_key: tuple[int, int]) -> str:
    video, tile = content_key
    if tile == -1:
        return f"V{video}-Base"
    return f"V{video}-T{tile}"

def _snapshot_cache(env) -> set[tuple[int, int]]:
    cache_set: set[tuple[int, int]] = set()
    cache_entries = getattr(env.mec_cache.policy, "cache", [])

    for video, tile in cache_entries:
        if int(video) == -1:
            continue
        cache_set.add((int(video), int(tile)))

    return cache_set

def plot_topk_requested_with_cache(
    request_counts: dict[tuple[int, int], int],
    cache_snapshot: set[tuple[int, int]],
    episode: int,
    top_k: int,
    out_dir: str,
) -> None:
    if not request_counts:
        return

    sorted_items = sorted(
        request_counts.items(),
        key=lambda kv: kv[1],
        reverse=True,
    )[:max(1, int(top_k))]

    labels = [_content_label(key) for key, _ in sorted_items]
    values = [count for _, count in sorted_items]
    colors = ["#2ca02c" if key in cache_snapshot else "#1f77b4" for key, _ in sorted_items]

    fig, ax = plt.subplots(figsize=(max(10, len(labels) * 0.8), 5))
    bars = ax.bar(labels, values, color=colors)

    ax.set_title(f"Episode {episode}: Top-{len(sorted_items)} Requested Content")
    ax.set_xlabel("Content")
    ax.set_ylabel("Request count")
    ax.tick_params(axis="x", rotation=35)

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            str(value),
            ha="center",
            va="bottom",
            fontsize=9,
        )

    from matplotlib.patches import Patch

    ax.legend(
        handles=[
            Patch(facecolor="#2ca02c", label="In cache (episode end)"),
            Patch(facecolor="#1f77b4", label="Not in cache"),
        ]
    )

    plt.tight_layout()
    os.makedirs(out_dir, exist_ok=True)
    plot_path = os.path.join(out_dir, f"topk_requested_ep{episode}.png")
    fig.savefig(plot_path, dpi=150)
    plt.close(fig)

In [ ]:
def process_missing_layer(
    agent,
    req_state,
    env,
    net_adapter,
    missing,
    transition,
    backhaul_usage: int,
    layer_idx: int,
    tile_idx: int | None,
    is_base: bool,
    bytes_cost: int
) -> int:
    if missing[layer_idx] != 1:
        return backhaul_usage

    cache = env.mec_cache.policy.cache

    if (-1, -1) in cache:
        action_idx = cache.index((-1, -1))

        message = {
            "video": req_state["video"],
            "tiles": [] if tile_idx is None else [req_state["viewport"][tile_idx]],
            "base_req_init": is_base,
            "action_idx": action_idx + 1,
        }
        env.prefetch_fn(env.mec_cache, message)

        backhaul_usage += bytes_cost
        missing[layer_idx] = 0

        return backhaul_usage

    if tile_idx is None:
        state, act_idx = net_adapter.build_observation(None, req_state["video"])
        tiles = []
    else:
        tile = req_state["viewport"][tile_idx]
        state, act_idx = net_adapter.build_observation(tile_idx, req_state["video"], tile)
        tiles = [tile]

    selection = agent.select_action(state)

    action = selection[0]
    value = selection[1] if len(selection) > 1 else None
    prob = selection[2] if len(selection) > 2 else None
    entropy = selection[3] if len(selection) > 3 else None

    if not is_base:
        pos_base_layer = cache.index((req_state["video"], -1))

        if act_idx[action] == pos_base_layer + 1:
            return backhaul_usage

    message = {
        "video": req_state["video"],
        "tiles": tiles,
        "base_req_init": is_base,
        "action_idx": act_idx[action],
    }
    env.prefetch_fn(env.mec_cache, message)

    if action != 0:
        backhaul_usage += bytes_cost

    transition[layer_idx] = {
        "state": state,
        "action": action,
        "value": value,
        "prob": prob,
        "entropy": entropy,
    }
    
    return backhaul_usage


def select_action(agent, req_state, env, net_adapter=None):
    if req_state is None:
        return None, np.zeros(5, dtype=np.int32), [None] * 5

    backhaul_usage = 0
    transition = [None] * 5
    missing = env._missing_items(req_state)

    backhaul_usage = process_missing_layer(
        agent=agent,
        req_state=req_state,
        env=env,
        net_adapter=net_adapter,
        missing=missing,
        transition=transition,
        backhaul_usage=backhaul_usage,
        layer_idx=0,
        tile_idx=None,
        is_base=True,
        bytes_cost=12 * env.mec_cache.tile_size_bytes[0]
    )

    has_base_layer = (req_state["video"], -1) in env.mec_cache.policy.cache
    if has_base_layer:
        for idx in range(cfg.viewport):
            backhaul_usage = process_missing_layer(
                agent=agent,
                req_state=req_state,
                env=env,
                net_adapter=net_adapter,
                missing=missing,
                transition=transition,
                backhaul_usage=backhaul_usage,
                layer_idx=idx + 1,
                tile_idx=idx,
                is_base=False,
                bytes_cost=env.mec_cache.tile_size_bytes[1]
            )

    log_selection_debug(transition, missing, backhaul_usage)
    return None, missing, transition


def _mean_video_psnr(video_psnr_sums: dict[int, float], video_psnr_counts: dict[int, int]) -> float:
    per_video_psnr = [
        video_psnr_sums[video_id] / video_psnr_counts[video_id]
        for video_id in video_psnr_counts
        if video_psnr_counts[video_id] > 0
    ]
    return float(np.mean(per_video_psnr)) if per_video_psnr else 0.0


def _popularity_bonus(
    video_id: int,
    video_watch_counts: dict[int, int],
    top_k: int,
    scale: float,
) -> float:
    if video_id not in video_watch_counts:
        return 0.0

    ranked_videos = sorted(
        video_watch_counts.items(),
        key=lambda item: item[1],
        reverse=True,
    )
    top_videos = [video for video, _ in ranked_videos[:max(1, int(top_k))]]
    if video_id not in top_videos:
        return 0.0

    rank = top_videos.index(video_id) + 1
    rank_weight = (max(1, int(top_k)) - rank + 1) / max(1, int(top_k))
    return float(scale) * rank_weight * float(np.log1p(video_watch_counts[video_id]))


def _print_video_psnr_ranking(
    video_watch_counts: dict[int, int],
    video_psnr_sums: dict[int, float],
    video_psnr_counts: dict[int, int],
    top_k: int,
) -> None:
    if not video_watch_counts:
        print("No watched videos recorded for this episode.")
        return

    ranked_videos = sorted(
        video_watch_counts.items(),
        key=lambda item: item[1],
        reverse=True,
    )[:max(1, int(top_k))]

    print(f"Mean PSNR across watched videos: {_mean_video_psnr(video_psnr_sums, video_psnr_counts):.2f}")
    print(f"Top {len(ranked_videos)} watched videos by rank:")
    for rank, (video_id, watch_count) in enumerate(ranked_videos, start=1):
        mean_psnr = 0.0
        if video_psnr_counts.get(video_id, 0) > 0:
            mean_psnr = video_psnr_sums[video_id] / video_psnr_counts[video_id]
        print(
            f"{rank:02d}. Video {video_id} | watches={watch_count} | mean_psnr={mean_psnr:.2f}"
        )


def run_episode(episode, env, agent, net_adapter, cfg, metrics_dir, global_step_start):
    """Run one full training episode and persist step-level metrics."""
    _, info = net_adapter.reset()

    total_reward = 0.0
    cache_hits = cache_misses = 0
    base_hits = base_misses = 0
    enh_hits = enh_misses = 0
    psnr_sum = 0.0
    popularity_bonus_sum = 0.0

    episode_request_counts: dict[tuple[int, int], int] = defaultdict(int)
    video_watch_counts: dict[int, int] = defaultdict(int)
    video_psnr_sums: dict[int, float] = defaultdict(float)
    video_psnr_counts: dict[int, int] = defaultdict(int)

    global_step = global_step_start

    if cfg.has_warmup and episode == 0:
        env.warmup_phase(net_adapter, 1000)

    for step in range(cfg.max_steps):
        global_step += 1
        req_state = info.get("user_request", None)

        current_video_id = None
        popularity_bonus = 0.0

        if req_state is not None:
            current_video_id = int(req_state["video"])
            episode_request_counts[_content_key(current_video_id, None)] += 1
            video_watch_counts[current_video_id] += 1
            for tile in req_state.get("viewport", []):
                episode_request_counts[_content_key(current_video_id, int(tile))] += 1

            popularity_bonus = _popularity_bonus(
                video_id=current_video_id,
                video_watch_counts=video_watch_counts,
                top_k=top_k,
                scale=popularity_reward_scale,
            )
            popularity_bonus_sum += popularity_bonus

        # --- Action Selection ---
        action, missing, transition = select_action(
            agent, req_state, env, net_adapter
        )

        # --- Environment Step ---
        _, reward, done, info = env.step(action, req_state, net_adapter)

        # --- Store Transition & Train ---
        nxt_req = info["user_request"]

        queued_update = False
        if missing[0] == 1 or transition[0] is not None:
            l0 = info["reward_layer_0"]

            state, _ = net_adapter.build_observation(None, nxt_req["video"])
            agent.remember(
                transition[0]["state"],
                transition[0]["action"],
                l0 + popularity_bonus,
                state,
                done,
            )
            queued_update = True

        for i in range(len(missing) - 1):
            if missing[i + 1] == 1 and transition[i + 1] is not None:
                l0 = info["reward_layer_0"]
                l1_e = info["reward_layer_1_details"][i] / cfg.viewport

                state, _ = net_adapter.build_observation(
                    i, nxt_req["video"], nxt_req["viewport"][i]
                )
                agent.remember(
                    transition[i + 1]["state"],
                    transition[i + 1]["action"],
                    l0 + l1_e + popularity_bonus,
                    state,
                    done,
                )
                queued_update = True

        if current_video_id is not None:
            step_psnr = float(info.get("psnr", 0.0))
            video_psnr_sums[current_video_id] += step_psnr
            video_psnr_counts[current_video_id] += 1

        if queued_update:
            train_metrics = agent.train_step()
            if train_metrics is not None:
                save_step_metrics(
                    metrics_dir=metrics_dir,
                    episode=episode,
                    episode_step=step,
                    global_step=global_step,
                    reward=reward,
                    agent=agent,
                    train_metrics=train_metrics,
                )

        delta_r, bs_hits, bs_miss, e_hits, e_miss = update_metrics(info, reward)
        total_reward += delta_r
        cache_hits += bs_hits + e_hits
        cache_misses += bs_miss + e_miss
        base_hits += bs_hits
        base_misses += bs_miss
        enh_hits += e_hits
        enh_misses += e_miss
        psnr_sum += info.get("psnr", 0.0)

        if done:
            break

        debugger.log("cache_hits", cache_hits)
        debugger.log("cache_misses", cache_misses)

    cache_snapshot = _snapshot_cache(env)

    return (
        total_reward,
        cache_hits,
        cache_misses,
        base_hits,
        base_misses,
        enh_hits,
        enh_misses,
        global_step,
        psnr_sum / (step + 1),
        dict(episode_request_counts),
        dict(video_watch_counts),
        dict(video_psnr_sums),
        dict(video_psnr_counts),
        cache_snapshot,
        popularity_bonus_sum,
    )


def train(cfg):
    env = builders.build_environment(cfg)
    agent = A2CWorker(cfg, debugger=debugger)
    feature_adapter = FeatureAdapter(cfg, env)
    net_adapter = NetworkAdapter(cfg, env, feature_adapter)

    date_dir = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M")
    debug_path = os.path.join(cfg.path_results, date_dir)
    metrics_dir = os.path.join(debug_path, "metrics")
    os.makedirs(metrics_dir, exist_ok=True)

    top_k = int(getattr(cfg, "topk_content_plot_k", 20))

    print(f"Starting training for {cfg.n_episodes} episodes... {date_dir}")
    print(f"Warmup Phase: {'Enabled' if cfg.has_warmup else 'Disabled'}")
    print(f"Users Session Length: {cfg.user_session_length}")
    print(agent)

    global_log_path = os.path.join(cfg.path_results, "global.log")
    with open(global_log_path, "a", encoding="utf-8") as f:
        f.write(f"{pd.Timestamp.now().isoformat()} | Max Steps: {cfg.max_steps} | ")
        f.write(f"{agent}\n")
        f.write("-" * 50 + "\n")

    global_step = 0

    for episode in range(cfg.n_episodes):
        (
            total_reward,
            cache_hits,
            cache_misses,
            base_hits,
            base_misses,
            enh_hits,
            enh_misses,
            global_step,
            psnr_rate,
            episode_request_counts,
            episode_video_watch_counts,
            episode_video_psnr_sums,
            episode_video_psnr_counts,
            cache_snapshot,
            popularity_bonus_sum,
        ) = run_episode(episode, env, agent, net_adapter, cfg, metrics_dir, global_step)

        agent.update_epsilon()

        save_episode_metrics(
            metrics_dir=metrics_dir,
            ep=episode,
            total_reward=total_reward,
            cache_hits=cache_hits,
            cache_misses=cache_misses,
            agent=agent,
        )

        debugger.save_results(filepath=f"{debug_path}/json/debug_ep{episode}")
        debugger.clear()

        base_hit_rate = base_hits / (base_hits + base_misses + 1e-9)
        enh_hit_rate = enh_hits / (enh_hits + enh_misses + 1e-9)
        mean_video_psnr = _mean_video_psnr(episode_video_psnr_sums, episode_video_psnr_counts)

        print(
            f"Episode {episode} | R: {int(total_reward)} | "
            f"HR: {cache_hits / (cache_hits + cache_misses + 1e-9):.2f} | "
            f"HR_2: {(base_hit_rate + enh_hit_rate) / 2:.2f} | "
            f"BHR: {base_hit_rate:.2f} | "
            f"EHR: {enh_hit_rate:.2f} | "
            f"PSNR: {psnr_rate:.2f} | "
            f"Mean video PSNR: {mean_video_psnr:.2f} | "
            f"Popularity bonus: {popularity_bonus_sum:.4f} | "
            f"Time: {pd.Timestamp.now().strftime('%H:%M:%S')}"
        )

        _print_video_psnr_ranking(
            video_watch_counts=episode_video_watch_counts,
            video_psnr_sums=episode_video_psnr_sums,
            video_psnr_counts=episode_video_psnr_counts,
            top_k=top_k,
        )
        print("-" * 50)

if __name__ == "__main__":
    train(cfg)


Starting training for 400 episodes... 2026-05-02_22-29
Warmup Phase: Enabled
Users Session Length: 30
A2CWorker(ActorLR=0.001000, CriticLR=0.002500)
BatchSize=128, Gamma=0.99
BufferSize=2000, GAE_lambda=0.95, Entropy_beta=0.01
Advantage_clip=5.0, Gradient_clip_norm=0.5
hiddens=(512,), Action Dim=21, State Dim=47
Episode 0 | R: 233310 | HR: 0.85 | HR_2: 0.73 | BHR: 0.96 | EHR: 0.49 | PSNR: 35.97 | Mean video PSNR: 35.92 | Popularity bonus: 9332.9246 | Time: 22:29:28
Mean PSNR across watched videos: 35.92
Top 20 watched videos by rank:
01. Video 0 | watches=547 | mean_psnr=35.78
02. Video 1 | watches=505 | mean_psnr=36.73
03. Video 2 | watches=317 | mean_psnr=36.31
04. Video 3 | watches=164 | mean_psnr=35.72
05. Video 9 | watches=106 | mean_psnr=35.83
06. Video 4 | watches=94 | mean_psnr=35.43
07. Video 17 | watches=88 | mean_psnr=36.68
08. Video 5 | watches=86 | mean_psnr=35.15
09. Video 8 | watches=85 | mean_psnr=36.21
10. Video 28 | watches=84 | mean_psnr=37.77
11. Video 18 | watches=

KeyboardInterrupt: 